# PalmVision AI — Training YOLO26 (SawitMVC)
Notebook ini menggantikan notebook tugas sebelumnya. Perubahan utama:

1. **Kelas dipetakan ulang** ke kategori kematangan yang bermakna (lihat sel konfigurasi). B1–B4 di SawitMVC adalah *perkiraan bulan menuju panen*, bukan mentah/matang/lewat-matang.
2. **Model lebih besar + resolusi lebih tinggi** (yolo26s, 960 px) dan training sampai konvergen, bukan 10 epoch.
3. **Augmentasi aman untuk warna** (hue tidak digeser, karena warna = sinyal kematangan).
4. **Threshold per kelas** dihitung dari data *validation* (bukan test), lalu disimpan untuk aplikasi.
5. **Test set dipakai sekali** untuk laporan akhir.
6. **Export ONNX** + `model_meta.json` siap diletakkan di `frontend/model/`.

Runtime: Colab → Runtime → Change runtime type → GPU (T4).

In [ ]:
!pip install -q -U ultralytics huggingface_hub pandas pyyaml onnx onnxslim onnxruntime

In [ ]:
import os, re, json, shutil, random, datetime
from pathlib import Path
import numpy as np, pandas as pd, yaml, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

# ---- Simpan hasil ke Google Drive supaya aman kalau sesi Colab putus ----
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/PalmVision"); OUT.mkdir(parents=True, exist_ok=True)

## Konfigurasi

**Pemetaan kelas SawitMVC → kategori aplikasi.** Menurut dokumentasi dataset, B1–B4 adalah kelas BBC berdasarkan perkiraan bulan menuju panen
(B1 ≤1 bln, B2 ~2 bln, B3 ~3 bln, B4 ~4 bln). Dataset **tidak punya kelas "terlalu matang"**.

| SawitMVC | Ciri | Kelas aplikasi |
|---|---|---|
| B1 | merah, besar, bulat | `matang` |
| B2 | hitam dengan transisi merah | `menjelang_matang` |
| B3 | hitam pekat, berduri | `mentah` |
| B4 | paling kecil, hijau-hitam | `mentah` |

B3 dan B4 digabung karena keduanya paling sering tertukar (di notebook lama: 204 B3 dikira B4) dan keduanya sama-sama "belum waktunya panen".

Untuk menambah `terlalu_matang`, kamu butuh dataset lain yang memang memiliki kelas itu (mis. dataset TBS pabrik dari Suharjito dkk. 2023 atau dataset Malaysia 2024). Gabungkan ke folder data, lalu tambahkan nama kelasnya di `CLASSES`.

In [ ]:
MODEL      = "yolo26s.pt"   # n=ringan, s=seimbang (default), m=lebih akurat tapi lebih lambat di browser
IMGSZ      = 960            # objek kecil -> resolusi lebih tinggi dari 640
EPOCHS     = 60
BATCH      = 8
PATIENCE   = 20
RUN_NAME   = "palm3_y26s_960"

CLASSES = ["mentah", "menjelang_matang", "matang"]          # urutan = id kelas (ordinal)
CLASS_MAP = {0: 2, 1: 1, 2: 0, 3: 0}                          # id SawitMVC (B1..B4) -> id baru
COLORS  = {"mentah": "#2e9e5b", "menjelang_matang": "#e0a100", "matang": "#d6342c"}

In [ ]:
from huggingface_hub import snapshot_download
ROOT = Path("/content/SawitMVC")
snapshot_download(repo_id="ULM-DS-Lab/SawitMVC", repo_type="dataset", local_dir=str(ROOT),
                  allow_patterns=["images/*", "labels/*", "split_manifest.csv", "data.yaml", "json/*"])
manifest = pd.read_csv(ROOT / "split_manifest.csv")
print(manifest.new_split.value_counts())

In [ ]:
# Split per POHON (bukan per foto) supaya foto pohon yang sama tidak bocor antara train/val/test.
BASE = Path("/content/palm3"); shutil.rmtree(BASE, ignore_errors=True)
for s in ["train", "val", "test"]:
    (BASE/"images"/s).mkdir(parents=True); (BASE/"labels"/s).mkdir(parents=True)

split_map = dict(zip(manifest.tree_id.astype(str), manifest.new_split.astype(str).str.lower()))
counts = {"train":0, "val":0, "test":0}; inst = {s: np.zeros(len(CLASSES), int) for s in counts}

for img in (ROOT/"images").glob("*.jpg"):
    split = split_map.get(re.sub(r"_\d+$", "", img.stem))
    split = "val" if split == "validation" else split
    lab = ROOT/"labels"/f"{img.stem}.txt"
    if split not in counts or not lab.exists(): continue
    lines = []
    for ln in lab.read_text().splitlines():
        p = ln.split()
        if len(p) != 5: continue
        new = CLASS_MAP[int(p[0])]
        inst[split][new] += 1
        lines.append(" ".join([str(new)] + p[1:]))
    shutil.copy2(img, BASE/"images"/split/img.name)
    (BASE/"labels"/split/lab.name).write_text("\n".join(lines) + ("\n" if lines else ""))
    counts[split] += 1

print("Gambar:", counts)
print(pd.DataFrame(inst, index=CLASSES))

(BASE/"data.yaml").write_text(yaml.safe_dump({
    "path": str(BASE), "train": "images/train", "val": "images/val", "test": "images/test",
    "names": {i: n for i, n in enumerate(CLASSES)}}, sort_keys=False))

## Training
Catatan augmentasi: `hsv_h` dikecilkan (warna adalah sinyal utama), `flipud=0` (posisi vertikal buah di pohon informatif), `mixup=0`.
`close_mosaic=15` memberi 15 epoch terakhir tanpa mosaic supaya model menyesuaikan diri dengan foto asli.
Waktu training bergantung GPU; kalau terlalu lama, turunkan `EPOCHS` atau pakai `MODEL="yolo26n.pt"`.

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data=str(BASE/"data.yaml"), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
    seed=SEED, cos_lr=True, close_mosaic=15, workers=2,
    hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, degrees=0.0, scale=0.4,
    fliplr=0.5, flipud=0.0, mixup=0.0,
    project=str(OUT/"runs"), name=RUN_NAME, exist_ok=True, plots=True,
)
BEST = OUT/"runs"/RUN_NAME/"weights"/"best.pt"
print(BEST, BEST.exists())

## Threshold per kelas (dari VALIDATION, bukan test)

In [ ]:
from ultralytics.utils.metrics import box_iou

best = YOLO(str(BEST))
nc = len(CLASSES)

def read_gt(label_path, w, h):
    boxes, cls = [], []
    if label_path.exists():
        for ln in label_path.read_text().splitlines():
            p = ln.split()
            if len(p) != 5: continue
            c = int(p[0]); cx, cy, bw, bh = map(float, p[1:])
            boxes.append([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h]); cls.append(c)
    return np.array(boxes).reshape(-1, 4), np.array(cls, dtype=int)

def collect(split, iou_thr=0.5):
    rec = {c: [] for c in range(nc)}; n_gt = {c: 0 for c in range(nc)}
    for r in best.predict(source=str(BASE/"images"/split), imgsz=IMGSZ, conf=0.05, iou=0.6,
                          batch=16, stream=True, verbose=False):
        h, w = r.orig_shape
        gb, gc = read_gt(BASE/"labels"/split/f"{Path(r.path).stem}.txt", w, h)
        pb = r.boxes.xyxy.cpu().numpy(); pc = r.boxes.cls.cpu().numpy().astype(int); ps = r.boxes.conf.cpu().numpy()
        for c in range(nc):
            g = gb[gc == c]; n_gt[c] += len(g)
            m = pc == c; b, s = pb[m], ps[m]; used = set()
            for i in np.argsort(-s):
                tp = 0
                if len(g):
                    ious = box_iou(torch.tensor(b[i:i+1]), torch.tensor(g))[0].numpy()
                    for j in used: ious[j] = -1
                    j = int(ious.argmax())
                    if ious[j] >= iou_thr: tp = 1; used.add(j)
                rec[c].append((float(s[i]), tp))
    return rec, n_gt

rec, n_gt = collect("val")
thr = {}
for c, name in enumerate(CLASSES):
    arr = np.array(rec[c]) if rec[c] else np.zeros((0, 2))
    best_f1, best_t = -1, 0.25
    for t in np.linspace(0.05, 0.95, 91):
        sel = arr[arr[:, 0] >= t] if len(arr) else arr
        tp = sel[:, 1].sum() if len(sel) else 0; fp = len(sel) - tp
        p = tp / (tp + fp) if tp + fp else 0; r_ = tp / n_gt[c] if n_gt[c] else 0
        f1 = 2*p*r_ / (p + r_) if p + r_ else 0
        if f1 > best_f1: best_f1, best_t = f1, float(t)
    thr[name] = round(best_t, 2)
    print(f"{name:18s} threshold={thr[name]:.2f}  F1(val)={best_f1:.3f}  GT={n_gt[c]}")

## Evaluasi akhir di TEST (satu kali)
Jangan ubah model/threshold lagi setelah melihat angka ini; kalau diubah, test tidak lagi independen.
Confusion matrix Ultralytics berorientasi **[prediksi, aktual]** (baris = prediksi, kolom = aktual), dengan *background* di indeks terakhir.
Gambar confusion matrix yang sudah benar tersimpan otomatis di folder hasil `val` di bawah (`plots=True`).

In [ ]:
m = best.val(data=str(BASE/"data.yaml"), split="test", imgsz=IMGSZ, conf=0.001, iou=0.6,
             plots=True, project=str(OUT/"runs"), name=RUN_NAME+"_test", exist_ok=True)
rows = []
for i, name in enumerate(CLASSES):
    p, r_, ap50, ap = m.box.class_result(i)
    rows.append({"kelas": name, "precision": p, "recall": r_, "mAP50": ap50, "mAP50-95": ap})
rows.append({"kelas": "SEMUA", "precision": m.box.mp, "recall": m.box.mr, "mAP50": m.box.map50, "mAP50-95": m.box.map})
test_df = pd.DataFrame(rows); print(test_df.round(4).to_string(index=False))

## Export ONNX + metadata untuk aplikasi

In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False, half=False)
onnx_path = Path(onnx_path); print(onnx_path, f"{onnx_path.stat().st_size/1e6:.1f} MB")

# cek cepat: bentuk output ONNX (YOLO26 end-to-end biasanya [1, 300, 6])
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print("input :", sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print("output:", sess.get_outputs()[0].name, sess.get_outputs()[0].shape)

In [ ]:
meta = {
    "version": f"{RUN_NAME}-{datetime.date.today().isoformat()}",
    "imgsz": IMGSZ,
    "review_margin": 0.15,
    "classes": [{"id": i, "name": n, "color": COLORS[n], "conf": thr[n]} for i, n in enumerate(CLASSES)],
    "dataset": "SawitMVC (ULM-DS-Lab), CC BY-NC 4.0 - hanya untuk penggunaan non-komersial",
    "class_map": "B1->matang, B2->menjelang_matang, B3+B4->mentah. Tidak ada kelas terlalu_matang.",
    "metrics_test": {r["kelas"]: {k: round(float(r[k]), 4) for k in ["precision", "recall", "mAP50", "mAP50-95"]} for r in rows},
}
(OUT/"model_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
shutil.copy2(onnx_path, OUT/"best.onnx")
print(json.dumps(meta, indent=2, ensure_ascii=False))

shutil.copy2(onnx_path, "/content/best.onnx"); shutil.copy2(OUT/"model_meta.json", "/content/model_meta.json")
from google.colab import files
files.download("/content/best.onnx"); files.download("/content/model_meta.json")

## Setelah selesai
1. Taruh `best.onnx` dan `model_meta.json` ke `frontend/model/` (timpa file placeholder).
2. Deploy ulang ke Vercel.
3. Ikuti `README.md` bagian *Meningkatkan akurasi*: koreksi di aplikasi tersimpan di Supabase dan bisa jadi data training berikutnya.